In [1]:
# Set working directory
import os
os.chdir("../../")

## Imports

In [2]:
import os
import numpy as np
import pandas as pd
import glob
import re

## Load data

In [5]:
# Configure file paths
sumprom_chec_glob = "sumproms/*.gz" # promoter-level signal
folders = ["prom_signals"] # nt-level promoter signals folder
prom_seqs = pd.read_csv('../../vovam/pipelines/new_prom_def_seqs.csv', index_col=0) # promoter location definitions
nuc_scores = pd.read_parquet('../../vovam/Yeast_General_Data/nucleosome_signals_per_promoter.parquet') # nt-level nucleosome scores
fimo = pd.read_csv('fimo_results/promoters/fimo.tsv', sep='\t')

In [6]:
sumprom_chec_files = glob.glob(sumprom_chec_glob)
sumprom_all = pd.concat([pd.read_parquet(x) for x in sumprom_chec_files], axis=1)

corr_cutoff = 0.895

def filter_reproducible(sumprom_all: pd.DataFrame, cutoff) -> pd.DataFrame:
    df = sumprom_all.copy()
    groups = pd.Series(df.columns, index=df.columns).str.rsplit("_", n=2).str[0]
    
    keep = []
    for _, members in groups.groupby(groups).groups.items():
        if len(members) < 2:
            continue
        corr = df[members].corr()
        np.fill_diagonal(corr.values, np.nan)
        max_corrs = corr.max(axis=1)
        reproducible = max_corrs[max_corrs >= cutoff].index.tolist()
        keep.extend(reproducible)
    return df[keep]

sumprom_filtered = filter_reproducible(sumprom_all, cutoff=corr_cutoff)

## Define motif families

In [8]:
FOXK1_WT = ["FOXP3","FOXA2","FOXF1","FOXL1","FOXL2","FOXJ2","FOXO3","FOXP1","FOXP2"]
FOX_DBD_swaps = ["FOXL2_DBD_FOXO3_IDR","FOXL2_DBD_FOXA2_IDR","FOXL2_DBD_FOXF1_IDR","FOXL2_DBD_FOXP3_IDR","FOXL2_DBD_FOXP1_IDR","FOXL2_DBD_FOXP2_IDR"]  
GABPA_WT = ["ELF1","ELF2","ERF1","ELK1","ELK4","ERG","FLI1"]
GABPA_DBD_swaps = ["ERG_DBD_ELK4","ERG_DBD_ELK1","ERG_DBD_FLI1","ERG_DBD_ERF","ERG_DBD_ELF1","ERG_DBD_ELF2","ERG_DBD","ERG_SAM_DBD","ELF1_DBD_ELK4","ELF1_DBD_ELK1","ELF1_DBD_FLI1","ELF1_DBD_ERF","ELF1_DBD_ELF2","ELF1_DBD_ERG","ELF1_DBD"]
SOX10_WT = ["SOX15","SOX17","SOX7","SOX11","SOX4","SOX6","SOX30","SOX9","SOX13","SOX5"]
SOX_DBD_IDR = ["SOX5_IDR","SOX6_IDR","SOX7_IDR","SOX9_IDR","SOX13_IDR","SOX17_IDR","SOX30_IDR","SOX5_DBD","SOX6_DBD","SOX7_DBD","SOX9_DBD","SOX13_DBD","SOX17_DBD","SOX30_DBD"]
HXD10_WT = ["CDX2","HOXA11","HOXC10","HOXD9","HOXA10","HOXC9","HOXA9","HOXB9","CDX4","HOXC13"]
GATA1_WT = ["GATA3","GATA6","GATA4","GATA2","GATA5"]
BATF3_WT = ["ATF4","FOS","CREB5","ATF1","CREB1","ATF2"]
HEY1_WT = ["MNT","MLXIPL","MLX","MXD4"]
NFAC4_WT = ["NFATC4","NFATC3"]
TF2LY_WT = ["TGIF2LX","TGIF2LY","TGIF1","TGIF2"]
PO3F2_WT = ["POU2F3","POU3F4","POU3F1"]

dbd_fam_dict = {"FOXK1_WT": FOXK1_WT, "FOX_DBD_swaps": FOX_DBD_swaps, "GABPA_WT": GABPA_WT, "GABPA_DBD_swaps": GABPA_DBD_swaps, "SOX10_WT": SOX10_WT, "SOX_DBD_IDR": SOX_DBD_IDR, "HXD10_WT": HXD10_WT, "GATA1_WT": GATA1_WT, "BATF3_WT": BATF3_WT, "HEY1_WT": HEY1_WT, "NFAC4_WT": NFAC4_WT, "TF2LY_WT": TF2LY_WT, "PO3F2_WT": PO3F2_WT}

# HOCOMOCO DBD-family representative motif
dbd_fam_representative = {"FOXK1_WT": 'FOXK1.H12CORE.0.PS.A', "FOX_DBD_swaps": 'FOXK1.H12CORE.0.PS.A', "GABPA_WT": 'GABPA.H12CORE.0.PSM.A', "GABPA_DBD_swaps": 'GABPA.H12CORE.0.PSM.A', "SOX10_WT": 'SOX10.H12CORE.1.PSM.A', "SOX_DBD_IDR": 'SOX10.H12CORE.1.PSM.A', "HXD10_WT": 'HXD10.H12CORE.0.SM.B', "GATA1_WT": 'GATA1.H12CORE.1.PSM.A', "BATF3_WT": 'BATF3.H12CORE.2.SM.B', "HEY1_WT": 'HEY1.H12CORE.0.S.B', "NFAC4_WT": 'NFAC4.H12CORE.1.SM.B', "TF2LY_WT": 'TF2LY.H12CORE.0.SM.B', "PO3F2_WT": 'PO3F2.H12CORE.2.SM.B'}

In [9]:
_RC = str.maketrans({"A":"T","T":"A","C":"G","G":"C","a":"t","t":"a","c":"g","g":"c"})
_VALID = set("ACGT")

def _base_key(col: str) -> str:
    p = col.split("_")
    return "_".join(p[:-2]) if len(p) >= 2 else col

def reps_for(sample_key: str, df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if _base_key(c) == sample_key]

def reverse_complement(seq: str) -> str:
    return seq.translate(_RC)[::-1]

In [10]:
# Resolve representative motif for a sample
def representative_motif_for_sample(sample: str,
                                    dbd_fam_dict: dict[str, list[str]],
                                    dbd_fam_representative: dict[str, str]) -> str | None:
    fam = next((fam for fam, samples in dbd_fam_dict.items() if sample in samples), None)
    if fam is None:
        return None
    return dbd_fam_representative.get(fam)

In [11]:
# Find highest quality HOCOMOCO motif for an individual TF
priority = {"A":0,"B":1,"C":2,"D":3}

def _safe_name(s): 
    return re.sub(r"[^A-Za-z0-9._+-]+","_",str(s))

def best_motif_by_prefix_from_fimo(fimo: pd.DataFrame):
    best = {}
    for mid in fimo["motif_id"].dropna().unique():
        pref = mid.split(".", 1)[0]
        m = re.search(r"\.([A-D])$", mid)
        rank = priority[m.group(1)] if m else 9
        if pref not in best or rank < best[pref][0]:
            best[pref] = (rank, mid)
    return {k: v for k, (r, v) in best.items()}

def _lookup_best_hoco(best_by_prefix: dict, hoco: str):
    if hoco in best_by_prefix: 
        return best_by_prefix[hoco]
    for k in best_by_prefix:
        if k.startswith(hoco + "_"):
            return best_by_prefix[k]
    return None

def motif_id_for_sample_hoco(sample: str, 
                             best_by_prefix: dict, 
                             genesymbol_to_hoco: dict, 
                             mismatched: dict | None = None):
    gene = sample.split("_", 1)[0]
    alt = (mismatched or {}).get(gene)
    for g in (gene, alt):
        if not g: 
            continue
        mid = _lookup_best_hoco(best_by_prefix, genesymbol_to_hoco.get(g, g))
        if mid: 
            return mid
    return None

# Converting strain name into associated HOCOMOCO gene symbol
genesymbol_to_hoco = {
 'ATF1': 'ATF1', 'ATF2': 'ATF2', 'ATF4': 'ATF4', 'CDX2': 'CDX2', 'CDX4': 'CDX4',
 'CREB1': 'CREB1', 'CREB5': 'CREB5', 'ELF1': 'ELF1', 'ELF1_DBD': 'ELF1',
 'ELF1_DBD_ELF2': 'ELF1', 'ELF1_DBD_ELK1': 'ELF1', 'ELF1_DBD_ELK4': 'ELF1',
 'ELF1_DBD_ERF': 'ELF1', 'ELF1_DBD_ERG': 'ELF1', 'ELF1_DBD_FLI1': 'ELF1',
 'ELF2': 'ELF2', 'ELK1': 'ELK1', 'ELK4': 'ELK4', 'ERF1': 'ERF', 'ERG': 'ERG',
 'ERG_DBD': 'ERG', 'ERG_DBD_ELF1': 'ERG', 'ERG_DBD_ELF2': 'ERG',
 'ERG_DBD_ELK1': 'ERG', 'ERG_DBD_ELK4': 'ERG', 'ERG_DBD_ERF': 'ERG',
 'ERG_DBD_FLI1': 'ERG', 'ERG_SAM_DBD': 'ERG', 'FLI1': 'FLI1', 'FOS': 'FOS',
 'FOXA2': 'FOXA2', 'FOXF1': 'FOXF1', 'FOXJ2': 'FOXJ2', 'FOXK1': 'FOXK1',
 'FOXL1': 'FOXL1', 'FOXL2': 'FOXL2',
 'FOXL2_DBD_FOXA2_IDR': 'FOXL2', 'FOXL2_DBD_FOXF1_IDR': 'FOXL2',
 'FOXL2_DBD_FOXO3_IDR': 'FOXL2', 'FOXL2_DBD_FOXP1_IDR': 'FOXL2',
 'FOXL2_DBD_FOXP2_IDR': 'FOXL2', 'FOXL2_DBD_FOXP3_IDR': 'FOXL2',
 'FOXO3': 'FOXO3', 'FOXP1': 'FOXP1', 'FOXP2': 'FOXP2', 'FOXP3': 'FOXP3',
 'GATA2': 'GATA2', 'GATA3': 'GATA3', 'GATA4': 'GATA4', 'GATA5': 'GATA5',
 'GATA6': 'GATA6', 'HOXA10': 'HXA10', 'HOXA11': 'HXA11', 'HOXA9': 'HXA9',
 'HOXB9': 'HXB9', 'HOXC10': 'HXC10', 'HOXC13': 'HXC13', 'HOXC9': 'HXC9',
 'HOXD9': 'HXD9', 'MLX': 'MLX', 'MLXIPL': 'MLXPL', 'MNT': 'MNT', 'MXD4': 'MAD4',
 'NFAT5': 'NFAT5', 'NFATC3': 'NFAC3', 'NFATC4': 'NFAC4', 'POU2F3': 'PO2F3',
 'POU3F1': 'PO3F1', 'POU3F4': 'PO3F4', 'SOX11': 'SOX11', 'SOX13': 'SOX13',
 'SOX13_DBD': 'SOX13', 'SOX13_IDR': 'SOX13', 'SOX15': 'SOX15', 'SOX17': 'SOX17',
 'SOX17_DBD': 'SOX17', 'SOX17_IDR': 'SOX17', 'SOX30': 'SOX30',
 'SOX30_DBD': 'SOX30', 'SOX30_IDR': 'SOX30', 'SOX4': 'SOX4', 'SOX5': 'SOX5',
 'SOX5_DBD': 'SOX5', 'SOX5_IDR': 'SOX5', 'SOX6': 'SOX6', 'SOX6_DBD': 'SOX6',
 'SOX6_IDR': 'SOX6', 'SOX7': 'SOX7', 'SOX7_DBD': 'SOX7', 'SOX7_IDR': 'SOX7',
 'SOX9': 'SOX9', 'SOX9_DBD': 'SOX9', 'SOX9_IDR': 'SOX9', 'TGIF1': 'TGIF1',
 'TGIF2': 'TGIF2', 'TGIF2LX': 'TF2LX', 'TGIF2LY': 'TF2LY'
}

In [12]:
# Create a switch to choose either the DBD-family representative motif or the TF's individual motif
def motif_id_for_sample(sample: str,
                        motif_mode: str,
                        fimo: pd.DataFrame,
                        dbd_fam_dict: dict[str, list[str]],
                        dbd_fam_representative: dict[str, str],
                        genesymbol_to_hoco_map: dict | None = None,
                        mismatched: dict | None = None) -> str | None:
    """
    motif_mode: 'representative_hoco' (default behavior) or 'individual_hoco'
    """
    if motif_mode == "individual_hoco":
        best_by_prefix = best_motif_by_prefix_from_fimo(fimo)
        return motif_id_for_sample_hoco(
            sample=sample,
            best_by_prefix=best_by_prefix,
            genesymbol_to_hoco=(genesymbol_to_hoco_map or genesymbol_to_hoco),
            mismatched=mismatched
        )
    # default: representative family motif
    return representative_motif_for_sample(sample, dbd_fam_dict, dbd_fam_representative)

## Footprint data extraction

In [13]:
# Replicate averaging
def _find_signal_file(rep: str, folders: list[str], suffix: str = "_signals.gz") -> str:
    for d in folders:
        p = os.path.join(d, f"{rep}{suffix}")
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Signal file for {rep} not found in {folders}")

def _load_signal(rep: str, folders: list[str]) -> pd.DataFrame:
    path = _find_signal_file(rep, folders)
    # Use parquet/CSV depending on your files; here we assume parquet if endswith .parquet
    if path.endswith(".gz"):
        return pd.read_parquet(path)
    # gz CSV with index in first column is also common; adapt if needed
    return pd.read_csv(path, index_col=0)

def average_sample_signals(sample: str,
                           sumprom_filtered: pd.DataFrame,
                           folders: list[str]) -> pd.DataFrame:
    reps = reps_for(sample, sumprom_filtered)
    if not reps:
        raise ValueError(f"No replicates found for {sample}")
    dfs = [_load_signal(r, folders) for r in reps]
    # Align rows/cols by index/columns before averaging
    base = dfs[0].copy()
    for d in dfs[1:]:
        base = base.add(d.reindex_like(base), fill_value=np.nan)
    return base / len(dfs)

In [14]:
# FIMO hits for a motif
def fimo_hits_for_motif(fimo: pd.DataFrame, motif_id: str) -> pd.DataFrame:
    cols = ["sequence_name", "start", "stop"] + (["strand"] if "strand" in fimo.columns else [])
    hits = fimo.loc[fimo["motif_id"].astype(str).str.strip() == str(motif_id), cols].copy()
    if hits.empty:
        return hits
    hits["sequence_name"] = hits["sequence_name"].astype(str)
    hits["start"] = pd.to_numeric(hits["start"], errors="coerce").astype("Int64")
    hits["stop"]  = pd.to_numeric(hits["stop"],  errors="coerce").astype("Int64")
    hits = hits.dropna(subset=["start", "stop"]).reset_index(drop=True)
    if "strand" in hits.columns:
        hits["strand"] = hits["strand"].astype(str).replace({"nan":"+"}).fillna("+")
    return hits

In [15]:
# Extract a ±flank window from a single track row-array (flip if strand=='-')
def _aligned_row(arr: np.ndarray) -> np.ndarray:
    # Skip leading NaNs so index 0 is first valid position
    mask = ~np.isnan(arr)
    if not mask.any():
        return np.array([], dtype=float)
    first = int(np.argmax(mask))
    return arr[first:]

def extract_window(track_df: pd.DataFrame,
                   seq_name: str,
                   center: int,
                   flank: int,
                   strand: str) -> np.ndarray | None:
    #Return length (2*flank+1) window or None if impossible. NaN-pad. Flip for '-'.
    if seq_name not in track_df.index:
        return None
    arr = pd.to_numeric(track_df.loc[seq_name], errors="coerce").to_numpy()
    aligned = _aligned_row(arr)
    if aligned.size == 0 or center < 0 or center >= aligned.size:
        return None
    a = max(0, center - flank)
    b = min(aligned.size, center + flank + 1)
    win = aligned[a:b]
    need = 2*flank + 1
    if win.size < need:
        lp = max(0, flank - center)
        rp = max(0, flank - (aligned.size - center - 1))
        win = np.pad(win, (lp, rp), constant_values=np.nan)
    if strand == "-":
        win = win[::-1]
    return win.astype(np.float32, copy=False)

In [16]:
# Sequence windows to match signal slicing (Series index=sequence_name)
# Return length (2*flank+1) sequence window (N-padded), reverse-complement if strand=='-'
def extract_sequence_window(seq_lookup: pd.Series | pd.DataFrame,
                            seq_name: str,
                            center: int,
                            flank: int,
                            strand: str) -> str:
    if isinstance(seq_lookup, pd.DataFrame):
        seq_lookup = seq_lookup["seq"].astype(str)
    full = str(seq_lookup.get(seq_name, "")).upper()
    need = 2*flank + 1
    if not full:
        return "N"*need
    a = max(0, center - flank)
    b = center + flank + 1
    # guard against short sequences (pad with N)
    left_pad = max(0, flank - center)
    right_pad = max(0, (center + flank + 1) - len(full))
    slice_ = full[a:min(b, len(full))] if b > a else ""
    if strand == "-":
        slice_ = reverse_complement(slice_)
    seq_win = ("N"*left_pad) + slice_ + ("N"*right_pad)
    if len(seq_win) < need:
        seq_win += "N"*(need - len(seq_win))
    return seq_win[:need]


In [17]:
# Cell 8: collapse near/identical hits
def _center_from_hit(start: int, stop: int) -> int:
    # 0-based center in the aligned arrays convention used above
    return int((int(start) + int(stop) - 2) // 2)

def collapse_hits(hits: pd.DataFrame,
                  raw_track: pd.DataFrame,
                  score_flank: int = 50,
                  min_separation: int = 10) -> pd.DataFrame:
    # 1) Greedy per sequence_name: drop hits whose start is within <min_separation of a kept start.
        # This is because FIMO sometimes identifies two motifs at the same site, when in fact it is just one motif
    # 2) Among remaining, collapse rows whose raw signal window (±score_flank) is exactly identical.
        # This is due to few cases of overlapping promoter definitions for genes on opposite strands
    
    if hits.empty:
        return hits

    # Step 1: proximity collapse (greedy by start)
    kept = []
    seen_by_seq: dict[str, list[int]] = {}
    hits_sorted = hits.sort_values(["sequence_name", "start"]).reset_index(drop=True)
    for h in hits_sorted.itertuples(index=False):
        seq = h.sequence_name; start = int(h.start)
        lst = seen_by_seq.setdefault(seq, [])
        if any(abs(start - ks) < min_separation for ks in lst):
            continue
        lst.append(start); kept.append(h)

    if not kept:
        return hits.iloc[0:0].copy()
    kept_df = pd.DataFrame(kept)

    # Step 2: raw-signal equality collapse (build key from ±score_flank window)
    def raw_key(row) -> tuple:
        seq = row.sequence_name
        strand = row.strand
        center = _center_from_hit(row.start, row.stop)
        win = extract_window(raw_track, seq, center, score_flank, strand)
        if win is None:
            return ("__missing__",)
        # Treat NaN as None so equality works
        return tuple(None if pd.isna(x) else float(x) for x in win)

    keys = {}
    rows = []
    for _, row in kept_df.iterrows():
        k = raw_key(row)
        if k in keys:
            continue
        keys[k] = True
        rows.append(row)

    return pd.DataFrame(rows).reset_index(drop=True)


In [18]:
# Cell 9: end-to-end extract for one sample
def extract_windows_for_sample(sample: str,
                               folders: list[str],
                               fimo: pd.DataFrame,
                               sumprom_filtered: pd.DataFrame,
                               dbd_fam_dict: dict[str, list[str]],
                               dbd_fam_representative: dict[str, str],
                               prom_seqs: pd.Series | pd.DataFrame,
                               nuc_scores: pd.DataFrame,
                               flank: int = 250,
                               collapse_min_sep: int = 10,
                               collapse_identical_flank: int = 50,
                               *,
                               motif_mode: str = "representative_hoco",
                               genesymbol_to_hoco_map: dict | None = None,
                               mismatched: dict | None = None,
                               raw_track_override: pd.DataFrame | None = None
                              ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    motif_id = motif_id_for_sample(
        sample=sample,
        motif_mode=motif_mode,
        fimo=fimo,
        dbd_fam_dict=dbd_fam_dict,
        dbd_fam_representative=dbd_fam_representative,
        genesymbol_to_hoco_map=genesymbol_to_hoco_map,
        mismatched=mismatched
    )
    if not motif_id:
        raise ValueError(f"No motif found for sample '{sample}'")
    print(f"{sample}: {motif_id}")

    if raw_track_override is not None:
        raw_track = raw_track_override
    else:
        raw_track = average_sample_signals(sample, sumprom_filtered, folders)

    nuc_track = nuc_scores

    hits = fimo_hits_for_motif(fimo, motif_id)
    if hits.empty:
        # Nothing to do
        return (pd.DataFrame(), pd.DataFrame())

    # Collapse hits by proximity & identical raw ±50bp
    hits2 = collapse_hits(hits, raw_track, score_flank=collapse_identical_flank,
                          min_separation=collapse_min_sep)
    if hits2.empty:
        return (pd.DataFrame(), pd.DataFrame())

    # Build windows
    need = 2*flank + 1
    cols = [f"pos_{i - flank}" for i in range(need)]
    raw_rows, nuc_rows, seq_rows, ids = [], [], [], []

    for h in hits2.itertuples(index=False):
        seq = h.sequence_name
        strand = h.strand
        center = _center_from_hit(h.start, h.stop)

        r = extract_window(raw_track, seq, center, flank, strand)
        n = extract_window(nuc_track, seq, center, flank, strand)
        s = extract_sequence_window(prom_seqs, seq, center, flank, strand)

        if r is None:  # skip entirely if we can't get raw (primary track)
            continue

        rid = f"{seq}_{int(h.start)}"
        ids.append(rid)
        raw_rows.append(r if r is not None else np.full(need, np.nan, np.float32))
        nuc_rows.append(n if n is not None else np.full(need, np.nan, np.float32))
        seq_rows.append(s)

    if not ids:
        return (pd.DataFrame(), pd.DataFrame())

    raw_df = pd.DataFrame(raw_rows, index=ids, columns=cols)
    nuc_df = pd.DataFrame(nuc_rows, index=ids, columns=cols)

    return raw_df, nuc_df


## Run the loop

In [20]:
# # Uncomment the loop to run

# out_dir = "footprint_data/hocomoco_default_pthreshold/individual_hoco_motif" # if motif_mode="individual_hoco"
# # out_dir = "footprint_data/hocomoco_default_pthreshold/representative_hoco_motif" # if motif_mode="individual_hoco"
# os.makedirs(out_dir, exist_ok=True)

# raw_dir = os.path.join(out_dir, "raw_signals")
# nuc_dir  = os.path.join(out_dir, "nuc_scores")
# for d in (raw_dir, nuc_dir):
#     os.makedirs(d, exist_ok=True)

# for fam, samples in dbd_fam_dict.items():
#     for sample in samples:
#         try:
#             raw_df, nuc_df = extract_windows_for_sample(
#                 sample=sample,
#                 folders=folders,
#                 fimo=fimo,
#                 sumprom_filtered=sumprom_filtered,
#                 dbd_fam_dict=dbd_fam_dict,
#                 dbd_fam_representative=dbd_fam_representative,
#                 prom_seqs=prom_seqs,
#                 nuc_scores=nuc_scores,
#                 flank=250,
#                 collapse_min_sep=10,
#                 collapse_identical_flank=50,
#                 motif_mode="individual_hoco", ## critical entry for choosing motif (dbd-family representative or own motif)
#                 genesymbol_to_hoco_map=genesymbol_to_hoco,
#             )
#             if raw_df.empty:
#                 print(f"skip {sample} (no data)")
#                 continue

#             raw_df.to_csv(os.path.join(raw_dir, f"{sample}_raw_windows.csv"))
#             nuc_df.to_csv(os.path.join(nuc_dir, f"{sample}_nucleosome_windows.csv"))
#             print(f"done {sample}")

#         except Exception as e:
#             print(f"skip {sample}: {e}")